# OceanEmbed — Local Data Cache Framework
> **SIH 2026 PS#01** | North Indian Ocean Subsurface Temperature Reconstruction
>
> **Run BLOCK 0 → BLOCK 4 once per session. Write your experiment in BLOCK 5.**
>
> Handles: credentials, downloads, caching, partial cache hits.
> You handle: everything else — preprocessing, models, embeddings, evaluation.
>
> Cache: `/run/media/mohan-krishna/vamSHE/data-pool/`

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# BLOCK 0 — IMPORTS · CREDENTIALS · PATHS
# ════════════════════════════════════════════════════════════════════════════

import os
import json
import uuid
import datetime
import pathlib
import shutil
import warnings
from datetime import date, timedelta

import numpy as np
import pandas as pd
import xarray as xr
import copernicusmarine
import earthaccess

warnings.filterwarnings("ignore")

# ── SSD storage paths ────────────────────────────────────────────────────────
DATA_ROOT  = pathlib.Path("/run/media/mohan-krishna/vamSHE/data-pool")
CACHE_ROOT = DATA_ROOT / "cache"
META_ROOT  = DATA_ROOT / "meta"
TMP_ROOT   = DATA_ROOT / "tmp"

for _p in [
    CACHE_ROOT / "sst",   CACHE_ROOT / "sss",    CACHE_ROOT / "ssh",
    CACHE_ROOT / "currents", CACHE_ROOT / "winds",
    CACHE_ROOT / "glorys", CACHE_ROOT / "argo",
    META_ROOT, TMP_ROOT,
]:
    _p.mkdir(parents=True, exist_ok=True)

# ── Credentials (plain text — private repo only) ─────────────────────────────
CMS_USERNAME  = "vardhangupthaimmidi@gmail.com"
CMS_PASSWORD  = "Hello@246812"
NASA_USERNAME = "vardhan_immidi"
NASA_PASSWORD = "Hello@246812"

print("✅ BLOCK 0 ready —", DATA_ROOT)
total, used, free = shutil.disk_usage(DATA_ROOT)
print(f"   💾 SSD: {free/1e9:.0f} GB free of {total/1e9:.0f} GB")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# BLOCK 1 — AUTH
# ════════════════════════════════════════════════════════════════════════════

copernicusmarine.login(
    username=CMS_USERNAME,
    password=CMS_PASSWORD,
    force_overwrite=True,
    check_credentials_valid=True,
)
print("✅ Copernicus Marine auth ready")

os.environ["EARTHDATA_USERNAME"] = NASA_USERNAME
os.environ["EARTHDATA_PASSWORD"] = NASA_PASSWORD
earthaccess.login(strategy="environment", persist=False)
print("✅ NASA EarthData auth ready")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# BLOCK 2 — CACHE ENGINE
#
# Architecture:
#   - Index stores independent *chunks*, each with its own time range + bbox.
#   - On every request: find spatially-covering chunks → compute uncovered
#     time gaps → download only gaps → merge all pieces → slice to window.
#   - Spatial rule: a chunk is a hit only if its stored bbox *contains*
#     the requested bbox (no 2-D spatial hole filling).
#   - Time: full partial-hit support — any overlap is reused.
# ════════════════════════════════════════════════════════════════════════════

CACHE_INDEX = META_ROOT / "cache_index.json"


def _idx_load() -> dict:
    if CACHE_INDEX.exists():
        return json.loads(CACHE_INDEX.read_text())
    return {}


def _idx_save(idx: dict):
    CACHE_INDEX.write_text(json.dumps(idx, indent=2))


def _bbox_contains(entry: dict,
                   lat_min: float, lat_max: float,
                   lon_min: float, lon_max: float) -> bool:
    """True if the cached chunk's bbox fully contains the requested bbox."""
    return (
        entry["lat_min"] <= lat_min and entry["lat_max"] >= lat_max and
        entry["lon_min"] <= lon_min and entry["lon_max"] >= lon_max
    )


def _chunks_covering(variable: str, product: str,
                     lat_min: float, lat_max: float,
                     lon_min: float, lon_max: float) -> list:
    """Return index entries that match variable + product and spatially cover the request."""
    return [
        e for e in _idx_load().values()
        if  e["variable"] == variable
        and e.get("product", "") == product
        and _bbox_contains(e, lat_min, lat_max, lon_min, lon_max)
        and pathlib.Path(e["file"]).exists()
    ]


def _find_gaps(chunks: list, start: str, end: str) -> list:
    """
    Given cached chunks (each with 'start'/'end' date strings),
    return list of (gap_start, gap_end) string pairs for uncovered periods.
    """
    req_s = date.fromisoformat(start)
    req_e = date.fromisoformat(end)

    intervals = []
    for c in chunks:
        c_s = date.fromisoformat(c["start"])
        c_e = date.fromisoformat(c["end"])
        if c_e < req_s or c_s > req_e:
            continue
        intervals.append((max(c_s, req_s), min(c_e, req_e)))

    if not intervals:
        return [(start, end)]

    intervals.sort()
    merged = [list(intervals[0])]
    for s, e in intervals[1:]:
        if s <= merged[-1][1] + timedelta(days=1):
            merged[-1][1] = max(merged[-1][1], e)
        else:
            merged.append([s, e])

    gaps = []
    cursor = req_s
    for s, e in merged:
        if cursor < s:
            gaps.append((cursor.isoformat(), (s - timedelta(days=1)).isoformat()))
        cursor = e + timedelta(days=1)
    if cursor <= req_e:
        gaps.append((cursor.isoformat(), req_e.isoformat()))

    return gaps


def _chunk_store(ds: xr.Dataset,
                 variable: str, product: str,
                 start: str, end: str,
                 lat_min: float, lat_max: float,
                 lon_min: float, lon_max: float,
                 extra: dict = None) -> str:
    """Save one chunk to SSD (via tmp → final) and register in the index."""
    chunk_id = uuid.uuid4().hex[:8]
    prefix   = f"{variable}_{start.replace('-','')}_{end.replace('-','')}"
    fp       = CACHE_ROOT / variable / f"{prefix}_{chunk_id}.nc"
    tmp      = TMP_ROOT / fp.name

    ds.to_netcdf(tmp)
    tmp.rename(fp)

    entry = {
        "variable": variable, "product":  product,
        "start":    start,    "end":      end,
        "lat_min":  lat_min,  "lat_max":  lat_max,
        "lon_min":  lon_min,  "lon_max":  lon_max,
        "file":     str(fp),
        "size_mb":  round(fp.stat().st_size / 1e6, 2),
        "cached_at": datetime.datetime.now().isoformat(),
        **(extra or {}),
    }
    idx = _idx_load()
    idx[chunk_id] = entry
    _idx_save(idx)
    print(f"  💾 STORED  [{variable}] {start}→{end}  → {fp.name}  ({entry['size_mb']} MB)")
    return chunk_id


def _cache_get_or_download(variable: str, product: str,
                            start: str, end: str,
                            lat_min: float, lat_max: float,
                            lon_min: float, lon_max: float,
                            download_fn,
                            force: bool = False) -> xr.Dataset:
    """
    Smart cache-aware fetch with partial time hit support.

    1. Find all cached chunks that spatially cover the requested bbox.
    2. Compute which time ranges are NOT covered (gaps).
    3. Download only the gaps via download_fn(start, end) -> xr.Dataset.
    4. Merge all relevant chunks on the time axis.
    5. Slice to exactly the requested [start, end] window and return.

    Parameters
    ----------
    download_fn : callable(start: str, end: str) -> xr.Dataset
        Called once per uncovered gap. Never called for cached ranges.
    force : bool
        If True, ignore cache and re-download everything.
    """
    if not force:
        chunks = _chunks_covering(variable, product, lat_min, lat_max, lon_min, lon_max)
        gaps   = _find_gaps(chunks, start, end)
    else:
        chunks, gaps = [], [(start, end)]

    if not gaps:
        print(f"  🟢 FULL HIT   [{variable}] {start}→{end}")
    else:
        for gap_s, gap_e in gaps:
            label = "🟡 PARTIAL" if chunks else "🔴 MISS"
            print(f"  {label}      [{variable}] downloading {gap_s}→{gap_e} ...")
            ds_gap = download_fn(gap_s, gap_e)
            _chunk_store(ds_gap, variable, product,
                         gap_s, gap_e,
                         lat_min, lat_max, lon_min, lon_max)

    chunks = _chunks_covering(variable, product, lat_min, lat_max, lon_min, lon_max)

    req_s = date.fromisoformat(start)
    req_e = date.fromisoformat(end)
    relevant = sorted(
        [
            c for c in chunks
            if date.fromisoformat(c["end"])   >= req_s
            and date.fromisoformat(c["start"]) <= req_e
        ],
        key=lambda c: c["start"],
    )

    if not relevant:
        raise RuntimeError(f"[{variable}] No data available after fetch — check download_fn.")

    if len(relevant) == 1:
        ds = xr.open_dataset(relevant[0]["file"])
    else:
        ds = xr.open_mfdataset(
            [c["file"] for c in relevant],
            combine="by_coords",
        )

    if "time" in ds.dims:
        ds = ds.sel(time=slice(start, end))

    return _norm_time(ds)


def cache_list():
    """Print the full cache inventory."""
    idx = _idx_load()
    if not idx:
        print("  (cache is empty)")
        return
    print(f"  {'Var':<12} {'Product':<42} {'Start':<12} {'End':<12} {'MB':>7}")
    print("  " + "-" * 90)
    for e in sorted(idx.values(), key=lambda x: (x["variable"], x["start"])):
        print(f"  {e['variable']:<12} {e.get('product',''):<42} "
              f"{e['start']:<12} {e['end']:<12} {e.get('size_mb', 0):>7.1f}")


def cache_invalidate(variable: str = None):
    """Remove cache entries and files. Pass variable name or None for all."""
    idx     = _idx_load()
    victims = [k for k, e in idx.items()
               if variable is None or e["variable"] == variable]
    for k in victims:
        fp = pathlib.Path(idx[k]["file"])
        if fp.exists():
            fp.unlink()
        del idx[k]
    _idx_save(idx)
    print(f"  🗑️  Removed {len(victims)} cache entries")


print("✅ BLOCK 2: Cache engine ready")
print("   cache_list()  ·  cache_invalidate(variable=None)")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# BLOCK 3 — DATA ACCESS FUNCTIONS
#
# All functions return raw xarray objects with unmodified native coordinates.
# No preprocessing, no interpolation, no depth selection, no grid alignment.
# All known dataset quirks are handled internally — documented per function.
# DO NOT MODIFY THIS BLOCK.
# ════════════════════════════════════════════════════════════════════════════


def _norm_time(obj):
    """
    Normalise 'time' coordinate to datetime64[ns].
    Handles cftime.DatetimeJulian, cftime.Datetime360Day, and any other
    non-standard calendar that xarray cannot natively slice or resample.
    Works on both xr.DataArray and xr.Dataset.
    """
    if "time" not in getattr(obj, "dims", {}):
        return obj
    try:
        t = pd.DatetimeIndex(obj.time.values).to_numpy().astype("datetime64[ns]")
    except Exception:
        t = np.array([np.datetime64(str(v)[:19], "ns") for v in obj.time.values])
    return obj.assign_coords(time=t)


# ══════════════════════════════════════════════════════════════════════════
# 1.  SST — Copernicus OSTIA  (0.05°, daily)
# ══════════════════════════════════════════════════════════════════════════
def get_sst(start: str, end: str,
            lat_min: float, lat_max: float,
            lon_min: float, lon_max: float,
            force: bool = False) -> xr.DataArray:
    """
    Sea Surface Temperature — Copernicus OSTIA.
    Returns xr.DataArray  shape (time, latitude, longitude)  units: °C
    Raw data is in Kelvin; converted to °C here.
    """
    PRODUCT = "METOFFICE-GLO-SST-L4-REP-OBS-SST"

    def _download(s, e):
        raw = copernicusmarine.open_dataset(
            dataset_id=PRODUCT, variables=["analysed_sst"],
            minimum_longitude=lon_min, maximum_longitude=lon_max,
            minimum_latitude=lat_min,  maximum_latitude=lat_max,
            start_datetime=s, end_datetime=e,
            username=CMS_USERNAME, password=CMS_PASSWORD,
        )
        da = raw["analysed_sst"] - 273.15
        da.attrs["units"] = "degC"
        return da.to_dataset(name="analysed_sst")

    return _cache_get_or_download(
        "sst", PRODUCT, start, end,
        lat_min, lat_max, lon_min, lon_max,
        _download, force=force,
    )["analysed_sst"]


# ══════════════════════════════════════════════════════════════════════════
# 2.  SSS — Copernicus SMAP/SMOS  (0.125°, daily)
# ══════════════════════════════════════════════════════════════════════════
def get_sss(start: str, end: str,
            lat_min: float, lat_max: float,
            lon_min: float, lon_max: float,
            force: bool = False) -> xr.DataArray:
    """
    Sea Surface Salinity — Copernicus SMAP/SMOS.
    Returns xr.DataArray  shape (time, latitude, longitude)  units: PSU
    BUG FIX: spurious depth=1 dimension returned by API — squeezed automatically.
    """
    PRODUCT = "cmems_obs-mob_glo_phy-sss_my_multi_P1D"

    def _download(s, e):
        raw = copernicusmarine.open_dataset(
            dataset_id=PRODUCT, variables=["sos"],
            minimum_longitude=lon_min, maximum_longitude=lon_max,
            minimum_latitude=lat_min,  maximum_latitude=lat_max,
            start_datetime=s, end_datetime=e,
            username=CMS_USERNAME, password=CMS_PASSWORD,
        )
        da = raw["sos"]
        if "depth" in da.dims:
            da = da.squeeze("depth", drop=True)   # BUG FIX
        return da.to_dataset(name="sos")

    return _cache_get_or_download(
        "sss", PRODUCT, start, end,
        lat_min, lat_max, lon_min, lon_max,
        _download, force=force,
    )["sos"]


# ══════════════════════════════════════════════════════════════════════════
# 3.  SSH — Copernicus DUACS  (0.25°, daily)
# ══════════════════════════════════════════════════════════════════════════
def get_ssh(start: str, end: str,
            lat_min: float, lat_max: float,
            lon_min: float, lon_max: float,
            force: bool = False) -> xr.DataArray:
    """
    Sea Level Anomaly — Copernicus DUACS.
    Returns xr.DataArray  shape (time, latitude, longitude)  units: m
    """
    PRODUCT = "c3s_obs-sl_glo_phy-ssh_my_twosat-l4-duacs-0.25deg_P1D"

    def _download(s, e):
        raw = copernicusmarine.open_dataset(
            dataset_id=PRODUCT, variables=["sla"],
            minimum_longitude=lon_min, maximum_longitude=lon_max,
            minimum_latitude=lat_min,  maximum_latitude=lat_max,
            start_datetime=s, end_datetime=e,
            username=CMS_USERNAME, password=CMS_PASSWORD,
        )
        return raw["sla"].to_dataset(name="sla")

    return _cache_get_or_download(
        "ssh", PRODUCT, start, end,
        lat_min, lat_max, lon_min, lon_max,
        _download, force=force,
    )["sla"]


# ══════════════════════════════════════════════════════════════════════════
# 4.  Currents — NASA OSCAR V2.0  (0.25°, 5-day)
# ══════════════════════════════════════════════════════════════════════════
def get_currents(start: str, end: str,
                 lat_min: float, lat_max: float,
                 lon_min: float, lon_max: float,
                 force: bool = False) -> xr.Dataset:
    """
    Surface geostrophic + Ekman currents — NASA OSCAR V2.0.
    Returns xr.Dataset  variables: u, v (m/s)  shape (time, latitude, longitude)
    ⚠  OSCAR cadence is 5-day, not daily.

    BUG FIXES applied internally:
    - Native dim order (time, longitude, latitude) transposed to (time, latitude, longitude)
    - cftime.DatetimeJulian normalised to datetime64[ns]
    - Non-dimension coord 'lat' alongside dim 'latitude' handled positionally
    """
    PRODUCT = "OSCAR_L4_OC_FINAL_V2.0"

    def _download(s, e):
        granules = earthaccess.search_data(
            short_name=PRODUCT, temporal=(s, e),
            bounding_box=(lon_min, lat_min, lon_max, lat_max),
        )
        if not granules:
            raise RuntimeError(f"No OSCAR granules found for {s}→{e}")
        ds = xr.open_mfdataset(
            earthaccess.open(granules), combine="by_coords",
        )[["u", "v"]]
        # BUG FIX: positional transpose handles 'lat' vs 'latitude' naming
        for var in ["u", "v"]:
            v    = ds[var]
            dims = list(v.dims)
            lp   = next((i for i, d in enumerate(dims) if "lat" in d.lower()), None)
            lop  = next((i for i, d in enumerate(dims) if "lon" in d.lower()), None)
            if lp is not None and lop is not None and lop < lp:
                ds[var] = v.transpose(dims[0], dims[lp], dims[lop])
        lat_d = "latitude" if "latitude" in ds.dims else "lat"
        lon_d = "longitude" if "longitude" in ds.dims else "lon"
        try:
            ds = ds.sel({lat_d: slice(lat_min, lat_max), lon_d: slice(lon_min, lon_max)})
        except Exception:
            pass
        return ds

    return _cache_get_or_download(
        "currents", PRODUCT, start, end,
        lat_min, lat_max, lon_min, lon_max,
        _download, force=force,
    )


# ══════════════════════════════════════════════════════════════════════════
# 5.  Winds — NASA CCMP V3.1  (0.25°, 6-hourly → daily mean)
# ══════════════════════════════════════════════════════════════════════════
def get_winds(start: str, end: str,
              lat_min: float, lat_max: float,
              lon_min: float, lon_max: float,
              force: bool = False) -> xr.Dataset:
    """
    10-m surface winds — NASA CCMP V3.1.
    Returns xr.Dataset  variables: uwnd, vwnd (m/s)  shape (time, latitude, longitude)
    Source is 6-hourly; resampled to daily mean before caching.

    BUG FIX: cftime normalised to datetime64[ns] BEFORE resample() —
    calling resample() on cftime time coords raises an exception.
    """
    PRODUCT = "CCMP_WINDS_10M6HR_L4_V3.1"

    def _download(s, e):
        granules = earthaccess.search_data(
            short_name=PRODUCT, temporal=(s, e),
            bounding_box=(lon_min, lat_min, lon_max, lat_max),
        )
        if not granules:
            raise RuntimeError(f"No CCMP granules found for {s}→{e}")
        ds = xr.open_mfdataset(
            earthaccess.open(granules), combine="by_coords",
        )[["uwnd", "vwnd"]]
        ds = _norm_time(ds)                    # BUG FIX: normalise BEFORE resample
        ds = ds.resample(time="1D").mean()
        lat_d = "latitude" if "latitude" in ds.dims else "lat"
        lon_d = "longitude" if "longitude" in ds.dims else "lon"
        try:
            ds = ds.sel({lat_d: slice(lat_min, lat_max), lon_d: slice(lon_min, lon_max)})
        except Exception:
            pass
        return ds

    return _cache_get_or_download(
        "winds", PRODUCT, start, end,
        lat_min, lat_max, lon_min, lon_max,
        _download, force=force,
    )


# ══════════════════════════════════════════════════════════════════════════
# 6.  GLORYS — Copernicus ocean reanalysis  (0.083°, daily, 3D)
# ══════════════════════════════════════════════════════════════════════════
def get_glorys(start: str, end: str,
               lat_min: float, lat_max: float,
               lon_min: float, lon_max: float,
               force: bool = False) -> xr.Dataset:
    """
    3D ocean temperature + salinity — Copernicus GLORYS12 reanalysis.
    Use as training TARGET for subsurface temperature reconstruction.

    Returns xr.Dataset  variables: thetao (°C), so (PSU)
                        shape: (time, depth, latitude, longitude)

    Depth: all 36 native GLORYS12 levels from 0.494 m to 1062.440 m.
    (minimum_depth=0, maximum_depth=1100 captures the 35 levels at/below
    1000 m plus the first level above at 1062.440 m.)
    Returned exactly as provided — no selection, no interpolation.
    """
    PRODUCT = "cmems_mod_glo_phy_my_0.083deg_P1D-m"

    def _download(s, e):
        return copernicusmarine.open_dataset(
            dataset_id=PRODUCT, variables=["thetao", "so"],
            minimum_longitude=lon_min, maximum_longitude=lon_max,
            minimum_latitude=lat_min,  maximum_latitude=lat_max,
            minimum_depth=0,           maximum_depth=1100,
            start_datetime=s, end_datetime=e,
            username=CMS_USERNAME, password=CMS_PASSWORD,
        )

    return _cache_get_or_download(
        "glorys", PRODUCT, start, end,
        lat_min, lat_max, lon_min, lon_max,
        _download, force=force,
    )


# ══════════════════════════════════════════════════════════════════════════
# 7.  ARGO — Gridded in-situ observations  (3-source fallback)
# ══════════════════════════════════════════════════════════════════════════
def get_argo(start: str, end: str,
             lat_min: float, lat_max: float,
             lon_min: float, lon_max: float,
             force: bool = False) -> xr.DataArray:
    """
    Gridded ARGO float temperature — for model evaluation.
    Returns xr.DataArray  temperature (°C)  shape (time, depth, latitude, longitude)
    Native depth levels returned as-is — no selection, no interpolation.

    Fallback order:
      1. Copernicus ARMOR3D  (cmems_obs-mob_glo_phy_my_0.125deg_P1D-m, 0.125°)
      2. INCOIS LAS THREDDS OPeNDAP  (public, no auth, may be slow)
      3. GLORYS12 thetao  (reanalysis fallback)
    """
    PRODUCT_ARMOR  = "cmems_obs-mob_glo_phy_my_0.125deg_P1D-m"
    PRODUCT_INCOIS = "INCOIS_LAS"

    def _download_armor(s, e):
        raw = copernicusmarine.open_dataset(
            dataset_id=PRODUCT_ARMOR, variables=["to"],
            minimum_longitude=lon_min, maximum_longitude=lon_max,
            minimum_latitude=lat_min,  maximum_latitude=lat_max,
            minimum_depth=0,           maximum_depth=1100,
            start_datetime=s, end_datetime=e,
            username=CMS_USERNAME, password=CMS_PASSWORD,
        )
        return raw["to"].to_dataset(name="to")

    try:
        print(f"  🌐 ARGO source 1: ARMOR3D [{start}→{end}]...")
        ds = _cache_get_or_download(
            "argo", PRODUCT_ARMOR, start, end,
            lat_min, lat_max, lon_min, lon_max,
            _download_armor, force=force,
        )
        return ds["to"]
    except Exception as exc:
        print(f"  ⚠️  ARMOR3D failed: {exc}")

    def _download_incois(s, e):
        url = (
            "https://las.incois.gov.in/thredds/dodsC/las/id-a43228e82a/"
            "data_home_las_datasets_argo_ValueAddedProducts.nc.jnl"
        )
        raw = xr.open_dataset(url, engine="pydap")
        var = list(raw.data_vars)[0]
        return raw[[var]]

    try:
        print(f"  🌐 ARGO source 2: INCOIS LAS...")
        ds = _cache_get_or_download(
            "argo", PRODUCT_INCOIS, start, end,
            lat_min, lat_max, lon_min, lon_max,
            _download_incois, force=force,
        )
        return _norm_time(ds[list(ds.data_vars)[0]])
    except Exception as exc:
        print(f"  ⚠️  INCOIS LAS failed: {exc}")

    print("  ℹ️  ARGO source 3: GLORYS thetao fallback")
    return get_glorys(start, end, lat_min, lat_max, lon_min, lon_max,
                      force=force)["thetao"]


print("✅ BLOCK 3: Data access functions ready")
print("   get_sst · get_sss · get_ssh · get_currents · get_winds · get_glorys · get_argo")
print("   All require: start, end, lat_min, lat_max, lon_min, lon_max")
print("   Optional:    force=True  (bypass cache and re-download)")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# BLOCK 4 — SANITY CHECK  (optional, non-destructive)
# ════════════════════════════════════════════════════════════════════════════

total, used, free = shutil.disk_usage(DATA_ROOT)
print(f"💾 SSD: {free/1e9:.0f} GB free of {total/1e9:.0f} GB")

print("\n📦 Cache inventory:")
cache_list()

try:
    results = earthaccess.search_data(
        short_name="OSCAR_L4_OC_FINAL_V2.0",
        count=1,
        temporal=("2020-01-01", "2020-01-05"),
    )
    print(f"\n✅ NASA EarthData: reachable ({len(results)} OSCAR granule found)")
except Exception as e:
    print(f"\n❌ NASA EarthData: {e}")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# BLOCK 5 — YOUR EXPERIMENT
#
# Available data functions:
#
#   get_sst(start, end, lat_min, lat_max, lon_min, lon_max)
#       → xr.DataArray (time, latitude, longitude)  °C
#         Source: Copernicus OSTIA  |  0.05°  |  daily
#
#   get_sss(start, end, lat_min, lat_max, lon_min, lon_max)
#       → xr.DataArray (time, latitude, longitude)  PSU
#         Source: Copernicus SMAP/SMOS  |  0.125°  |  daily
#
#   get_ssh(start, end, lat_min, lat_max, lon_min, lon_max)
#       → xr.DataArray (time, latitude, longitude)  m  (sea level anomaly)
#         Source: Copernicus DUACS  |  0.25°  |  daily
#
#   get_currents(start, end, lat_min, lat_max, lon_min, lon_max)
#       → xr.Dataset   (time, latitude, longitude)  u, v  m/s
#         Source: NASA OSCAR V2.0  |  0.25°  |  5-day cadence ⚠
#
#   get_winds(start, end, lat_min, lat_max, lon_min, lon_max)
#       → xr.Dataset   (time, latitude, longitude)  uwnd, vwnd  m/s
#         Source: NASA CCMP V3.1  |  0.25°  |  daily mean (from 6-hourly)
#
#   get_glorys(start, end, lat_min, lat_max, lon_min, lon_max)
#       → xr.Dataset   (time, depth, latitude, longitude)  thetao °C, so PSU
#         Source: Copernicus GLORYS12  |  0.083°  |  daily
#         36 native depth levels: 0.494 m → 1062.440 m  (all returned, no selection)
#
#   get_argo(start, end, lat_min, lat_max, lon_min, lon_max)
#       → xr.DataArray (time, depth, latitude, longitude)  °C
#         Source: ARMOR3D → INCOIS LAS → GLORYS (fallback chain)
#         Native depth levels returned as-is.
#
# Cache utilities:
#   cache_list()                — show all cached chunks
#   cache_invalidate("sst")     — remove cached variable (None = all)
#
# Caching behaviour:
#   - First call: downloads and caches the requested window as a chunk.
#   - Repeat call with same or narrower time / smaller bbox: instant hit.
#   - Wider time range: only the new dates are downloaded and merged.
#   - Different (larger) bbox: treated as a miss and re-downloaded for that bbox.
#   - force=True: bypass cache entirely and re-download.
# ════════════════════════════════════════════════════════════════════════════


